# Funciones comunes

Funciones reutilizables que los demás notebooks cargan con `%run`: fecha de ingesta, reemplazo de una partición y carga con `MERGE` en Delta.

### add_ingestion_date

Agrega la columna `ingestion_date` con la fecha y hora de ejecución.

In [0]:
def add_ingestion_date(df):

  from pyspark.sql.functions import current_timestamp
  
  return df.withColumn('ingestion_date', current_timestamp())


### overwrite_partitioned_table

Si la tabla existe, borra los registros de la partición indicada; se usaba antes de un `append` para evitar duplicados.

In [0]:
def overwrite_partitioned_table(schema_name, table_name, partition_column, partition_value):
    if spark.catalog.tableExists(f"{schema_name}.{table_name}"):
        
        spark.sql(f"DELETE FROM {schema_name}.{table_name} WHERE {partition_column} = '{partition_value}'")
        print(f"Se ha eliminado la partición {partition_value} de la tabla {schema_name}.{table_name}")
        
    else:
        print(f"{schema_name}.{table_name} no disponible")

### merge_delta_lake

Si la tabla existe, hace `MERGE` (actualiza coincidencias e inserta nuevos registros); si no, la crea particionada por la columna indicada.

In [0]:
def merge_delta_lake(input_df, schema_name, table_name, merge_condition, partition_column):

    from delta.tables import DeltaTable

    if spark.catalog.tableExists(f"{schema_name}.{table_name}"):
        deltaTable = DeltaTable.forName(spark, f"{schema_name}.{table_name}")

        deltaTable.alias('trgt') \
        .merge(
        input_df.alias('src'),
        merge_condition
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
        print(f"Se ha actualizado la tabla {schema_name}.{table_name}")

    else:
        (
        input_df.write
            .mode("overwrite")
            .partitionBy(f"{partition_column}")
            .saveAsTable(f"{schema_name}.{table_name}")
        )
        print(f"Se ha creado la tabla {schema_name}.{table_name}")